In [8]:
import pandas as pd
import seaborn as sn

Read CSV

In [9]:
file = "results_prefix_mem.csv"
df = pd.read_csv(file)
for c in df.columns:
    print(c, end=", ")
print("")
df.replace(' NaN', pd.NA, inplace=True)
print(f"{df.shape}")


app_useful_cycles, app_wasted_cycles, index, param_hs_enabled, param_hs_flags, param_jumbo_frames, param_lcores, param_n_rules, param_nb_rgx_desc, param_pattern_db, param_pkt_size, param_pktgen_latency_enabled, param_port_forward_mode, param_port_mp_size, param_port_rx_descriptors, param_port_tx_descriptors, param_rgx_enabled, param_rules_use_content_kw, param_rules_use_pcre_kw, param_target_tp, port0_rx_broadcast_bytes, port0_rx_broadcast_packets, port0_rx_errors, port0_rx_good_bytes, port0_rx_good_packets, port0_rx_mbuf_allocation_errors, port0_rx_missed_errors, port0_rx_multicast_bytes, port0_rx_multicast_packets, port0_rx_out_of_buffer, port0_rx_phy_bytes, port0_rx_phy_crc_errors, port0_rx_phy_discard_packets, port0_rx_phy_in_range_len_errors, port0_rx_phy_packets, port0_rx_phy_symbol_errors, port0_rx_prio0_buf_discard_packets, port0_rx_prio0_cong_discard_packets, port0_rx_prio1_buf_discard_packets, port0_rx_prio1_cong_discard_packets, port0_rx_prio2_buf_discard_packets, port0_rx_p

Find columns for hyperscan
Find columns for parameters

In [4]:
hs_cols = [c for c in df.columns if c.startswith("hs_")]
print(hs_cols, len(hs_cols))
param_cols = [c for c in df.columns if c.startswith("param_")]
print(param_cols, len(param_cols))
rgx_cols = [c for c in df.columns if c.startswith("rgx_")]
print(rgx_cols, len(rgx_cols))
general_cols = [c for c in df.columns if c not in hs_cols + param_cols + rgx_cols]
print(general_cols, len(general_cols))

[] 0
['param_hs_enabled', 'param_hs_flags', 'param_jumbo_frames', 'param_lcores', 'param_n_rules', 'param_nb_rgx_desc', 'param_pattern_db', 'param_pkt_size', 'param_pktgen_latency_enabled', 'param_port_forward_mode', 'param_port_mp_size', 'param_port_rx_descriptors', 'param_port_tx_descriptors', 'param_rgx_enabled', 'param_rules_use_content_kw', 'param_rules_use_pcre_kw', 'param_target_tp'] 17
['rgx_bs12_deq_avg_rtt_cycles', 'rgx_bs12_deq_avg_rtt_us', 'rgx_bs12_deq_brst_cnt', 'rgx_bs12_deq_brst_pkts_cnt', 'rgx_bs12_deq_rtt_ttl_cycles', 'rgx_bs12_deq_rtt_ttl_us', 'rgx_bs12_enq_brst_cnt', 'rgx_bs12_enq_brst_pkts_cnt', 'rgx_bs12_patterns_matched', 'rgx_bs16_deq_avg_rtt_cycles', 'rgx_bs16_deq_avg_rtt_us', 'rgx_bs16_deq_brst_cnt', 'rgx_bs16_deq_brst_pkts_cnt', 'rgx_bs16_deq_rtt_ttl_cycles', 'rgx_bs16_deq_rtt_ttl_us', 'rgx_bs16_enq_brst_cnt', 'rgx_bs16_enq_brst_pkts_cnt', 'rgx_bs16_patterns_matched', 'rgx_bs20_deq_avg_rtt_cycles', 'rgx_bs20_deq_avg_rtt_us', 'rgx_bs20_deq_brst_cnt', 'rgx_bs20

Isolate Hyperscan Results

In [30]:
hs_df = df[hs_cols + param_cols + general_cols]
hs_df = hs_df.dropna()
print(f"rows in hyperscan data: {hs_df.shape[0]}, rows in all data: {df.shape[0]}")

rows in hyperscan data: 1117, rows in all data: 1728


Isolate Regex results 

In [5]:
rgx_df = df[rgx_cols + param_cols + general_cols]
rgx_df = rgx_df.dropna()
print(f"rows in rgx data: {rgx_df.shape[0]}, rows in all data: {df.shape[0]}")
print(f"{rgx_df.shape=}")

rows in rgx data: 120, rows in all data: 120
rgx_df.shape=(120, 262)


Aggregate Regex Data according to Burst Sizes, three steps
First: find the measurements kind
Second: Cast them to the correct type
Third: Sum these measurements for all burst sizes

Step 1 and 2

In [6]:
rgx_measurements = list(set("_".join(c.split("_")[2:]) for c in rgx_cols))
rgx_df = rgx_df.astype({m:'float32' for m in rgx_cols})


In [7]:
rgx_df.head()

,rgx_bs12_deq_avg_rtt_cycles,rgx_bs12_deq_avg_rtt_us,rgx_bs12_deq_brst_cnt,rgx_bs12_deq_brst_pkts_cnt,rgx_bs12_deq_rtt_ttl_cycles,rgx_bs12_deq_rtt_ttl_us,rgx_bs12_enq_brst_cnt,rgx_bs12_enq_brst_pkts_cnt,rgx_bs12_patterns_matched,rgx_bs16_deq_avg_rtt_cycles,...,port0_tx_q6_bytes,port0_tx_q6_packets,port0_tx_unicast_bytes,port0_tx_unicast_packets,tx_p0_b2b_latency_cycles_avg,tx_p0_b2b_latency_cycles_max,tx_p0_b2b_latency_cycles_min,tx_p0_b2b_latency_us_avg,tx_p0_b2b_latency_us_max,tx_p0_b2b_latency_us_min
0,45764.0,216.0,6598.0,60710.0,2.778374e+09,13159570.0,60493.0,60493.0,0.0,88442.0,...,592200576,395856,4210779336,2820785,59157,657771,36823,19,219,12
1,43696.0,206.0,6836.0,62921.0,2.749432e+09,13017347.0,62791.0,62791.0,0.0,92043.0,...,592182624,395844,4210928016,2821070,60496,9613054,36115,20,3204,12
2,43880.0,207.0,11628.0,107080.0,4.698756e+09,22235012.0,106812.0,106812.0,0.0,88805.0,...,1183760864,791284,8417677232,5633150,55940,121115,36579,18,40,12
3,43935.0,207.0,11609.0,106926.0,4.697827e+09,22234068.0,106745.0,106745.0,0.0,79400.0,...,1184081008,791498,8417777328,5633314,58348,7866177,33398,19,2622,11
4,47345.0,223.0,12201.0,112284.0,5.316106e+09,25118260.0,112131.0,112131.0,0.0,45490.0,...,1774878336,1186416,12621463592,8443259,66077,22660480,33615,22,7553,11


In [8]:

rgx_df_agg = df[param_cols + general_cols]
for m in rgx_measurements:
    selected = [c for c in rgx_cols if c.endswith(m)]
    print(selected)
    col = f"tot_{m}"        
    rgx_df_agg[col] = rgx_df.loc[:, selected].sum(axis=1)
    print(f"Data for col {col}")
    print(rgx_df_agg.loc[:, selected].head(10))
    print(rgx_df_agg)
    break


['rgx_bs12_patterns_matched', 'rgx_bs16_patterns_matched', 'rgx_bs20_patterns_matched', 'rgx_bs24_patterns_matched', 'rgx_bs28_patterns_matched', 'rgx_bs32_patterns_matched', 'rgx_bs36_patterns_matched', 'rgx_bs40_patterns_matched', 'rgx_bs44_patterns_matched', 'rgx_bs48_patterns_matched', 'rgx_bs4_patterns_matched', 'rgx_bs52_patterns_matched', 'rgx_bs56_patterns_matched', 'rgx_bs60_patterns_matched', 'rgx_bs64_patterns_matched', 'rgx_bs8_patterns_matched']
Data for col tot_patterns_matched


/tmp/ipykernel_288958/1069659861.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rgx_df_agg[col] = rgx_df.loc[:, selected].sum(axis=1)


KeyError: "None of [Index(['rgx_bs12_patterns_matched', 'rgx_bs16_patterns_matched',\n       'rgx_bs20_patterns_matched', 'rgx_bs24_patterns_matched',\n       'rgx_bs28_patterns_matched', 'rgx_bs32_patterns_matched',\n       'rgx_bs36_patterns_matched', 'rgx_bs40_patterns_matched',\n       'rgx_bs44_patterns_matched', 'rgx_bs48_patterns_matched',\n       'rgx_bs4_patterns_matched', 'rgx_bs52_patterns_matched',\n       'rgx_bs56_patterns_matched', 'rgx_bs60_patterns_matched',\n       'rgx_bs64_patterns_matched', 'rgx_bs8_patterns_matched'],\n      dtype='object')] are in the [columns]"